# Generate Segmentation Labels dari BBOX pake SAM

Mengubah dataset bounding box kamu jadi polygon segmentation otomatis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Pastikan folder `dataset/` sudah diupload ke Google Drive (`/content/drive/MyDrive/dataset/`), berisi:
- `train/images/`
- `train/labels/`
- `val/images/`
- `val/labels/`

In [ ]:
!pip install ultralytics -q

In [ ]:
import os, cv2, numpy as np, shutil
from pathlib import Path
from ultralytics import SAM
from tqdm.notebook import tqdm

DATASET = Path("/content/drive/MyDrive/dataset")
model = SAM("mobile_sam.pt")

In [ ]:
def yolo_bbox_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = int((cx - w / 2) * img_w)
    y1 = int((cy - h / 2) * img_h)
    x2 = int((cx + w / 2) * img_w)
    y2 = int((cy + h / 2) * img_h)
    return [x1, y1, x2, y2]

def mask_to_polygon(mask, img_w, img_h):
    mask = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    epsilon = 0.001 * cv2.arcLength(cnt, True)
    cnt = cv2.approxPolyDP(cnt, epsilon, True)
    cnt = cnt.squeeze()
    if cnt.ndim != 2 or len(cnt) < 3:
        return None
    poly = []
    for x, y in cnt:
        poly.append(x / img_w)
        poly.append(y / img_h)
    return poly

In [ ]:
def process_split(split: str):
    img_dir = DATASET / split / "images"
    lbl_dir = DATASET / split / "labels"
    backup_dir = DATASET / split / "labels_detection"

    images = sorted(img_dir.glob("*"))
    print(f"Processing {split} - {len(images)} images")

    if not backup_dir.exists():
        backup_dir.mkdir()
        for f in lbl_dir.glob("*.txt"):
            shutil.copy2(f, backup_dir / f.name)
        print(f"  Backup -> {backup_dir}")

    failed = 0
    for img_path in tqdm(images):
        label_path = lbl_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        with open(label_path) as f:
            lines = f.read().strip().splitlines()

        seg_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = parts[0]
            cx, cy, w, h = map(float, parts[1:5])
            bbox = yolo_bbox_to_xyxy(cx, cy, w, h, img_w, img_h)

            try:
                results = model(img, bboxes=[bbox], verbose=False)[0]
            except:
                failed += 1
                continue

            if results.masks is None:
                failed += 1
                continue

            mask = results.masks.data[0].cpu().numpy()
            poly = mask_to_polygon(mask, img_w, img_h)
            if poly is None:
                failed += 1
                continue

            seg_lines.append(cls_id + " " + " ".join(f"{v:.6f}" for v in poly))

        with open(label_path, "w") as f:
            f.write("\n".join(seg_lines))

    print(f"Done {split}. Failed: {failed}")

process_split("train")
process_split("val")

Selesai! Label sekarang format segmentation. Label asli (detection) sudah di-backup ke `labels_detection/`. Silakan buka `colab_train.ipynb` dan Run All.